In [1]:
# Local environment check for QuimicaAutomocio P2.
# The original eChem notebooks are Colab-oriented; this one is designed for local use.
import numpy as np
import py3Dmol
import veloxchem as vlx

required = ["ScfRestrictedDriver", "OptimizationDriver", "VibrationalAnalysis"]
missing = [name for name in required if not hasattr(vlx, name)]
if missing:
    raise RuntimeError(f"VeloxChem is missing required classes: {missing}")

print("VeloxChem", getattr(vlx, "__version__", "unknown"))
print("Environment OK")

VeloxChem 1.0rc4
Environment OK


In [2]:
molecule = vlx.Molecule.read_smiles("OC(=O)[C@@H](N)Cc1ccc(N)cc1")

print(molecule.get_xyz_string())
molecule.show(atom_indices=True, width=520, height=360)

25

O             -3.692511000000        -2.628610000000        -0.352953000000
C             -3.095670000000        -1.728379000000         0.524361000000
O             -3.460884000000        -1.692179000000         1.730631000000
C             -2.002645000000        -0.822768000000         0.035438000000
N             -2.336649000000         0.566738000000         0.361982000000
C             -0.648010000000        -1.249672000000         0.640242000000
C              0.498760000000        -0.468348000000         0.051350000000
C              1.036521000000         0.633051000000         0.736962000000
C              2.083249000000         1.370734000000         0.175908000000
C              2.601415000000         1.020048000000        -1.078486000000
N              3.671497000000         1.775570000000        -1.652076000000
C              2.061433000000        -0.074325000000        -1.768207000000
C              1.014478000000        -0.812596000000        -1.208409000000
H       

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_drv.dispersion = True
scf_drv.ostream.mute()

opt_drv = vlx.OptimizationDriver(scf_drv)
opt_drv.ostream.mute()
opt_drv.max_iter = 100

opt_results = opt_drv.compute(molecule, basis)
opt_molecule = opt_results["final_molecule"]

print("Optimized geometry:")
print(opt_molecule.get_xyz_string())
print("Optimization steps:", len(opt_results.get("opt_energies", [])))
opt_molecule.show(atom_indices=True, width=520, height=360)

Optimized geometry:
25

O             -3.346749868411        -2.868366048915        -0.075153307685
C             -3.187382591554        -1.708819916962         0.591286156306
O             -3.802237012621        -1.440460581721         1.593640691920
C             -2.100175247423        -0.842493813604        -0.015108866595
N             -2.390351399437         0.540869330053         0.288289183765
C             -0.726443743158        -1.382486798593         0.501579245412
C              0.432624575893        -0.582097690778        -0.028535724420
C              0.951519493650         0.511330571991         0.680970304319
C              1.979988556050         1.297940941174         0.164397577180
C              2.533560118514         1.018056290980        -1.099081908547
N              3.523008964671         1.823301512728        -1.641925894492
C              2.018215444984        -0.080702717966        -1.815448840033
C              0.990306562428        -0.856188111234        -1.2

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
vib_drv = vlx.VibrationalAnalysis(scf_drv)
vib_drv.ostream.mute()
vib_drv.do_ir = True

vib_results = vib_drv.compute(opt_molecule, basis)

frequencies = np.array(vib_results["vib_frequencies"], dtype=float)
normal_modes = np.array(vib_results["normal_modes"], dtype=float)
reduced_masses = np.array(vib_results["reduced_masses"], dtype=float)
force_constants = np.array(vib_results["force_constants"], dtype=float)
ir_intensities = np.array(vib_results.get("ir_intensities", np.zeros_like(frequencies)), dtype=float)

print("mode  frequency / cm^-1   red. mass / amu   force const. / mdyn A^-1   IR / km mol^-1")
for i, freq in enumerate(frequencies, start=1):
    print(f"{i:>4d}  {freq:>16.2f}  {reduced_masses[i-1]:>15.4f}  {force_constants[i-1]:>25.4f}  {ir_intensities[i-1]:>14.2f}")

In [ ]:
def normal_mode_trajectory(molecule, mode, amplitude=0.45, frames=32):
    labels = list(molecule.get_labels())
    coords = np.array(molecule.get_coordinates_in_angstrom(), dtype=float)
    mode = np.array(mode, dtype=float)

    chunks = []
    for phase in np.linspace(0.0, 2.0 * np.pi, frames, endpoint=False):
        displaced = coords + amplitude * np.sin(phase) * mode
        chunks.append(str(len(labels)))
        chunks.append("normal-mode frame")
        for label, xyz in zip(labels, displaced):
            chunks.append(f"{label:2s} {xyz[0]: .8f} {xyz[1]: .8f} {xyz[2]: .8f}")
    return "\n".join(chunks) + "\n"

def show_mode(molecule, mode_index, amplitude=0.45, frames=32):
    traj = normal_mode_trajectory(molecule, normal_modes[mode_index], amplitude=amplitude, frames=frames)
    view = py3Dmol.view(width=650, height=430)
    view.addModelsAsFrames(traj, "xyz")
    view.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.24}})
    view.animate({"loop": "backAndForth", "reps": 0, "interval": 80})
    view.zoomTo()
    return view

# Python index: 0 is the first vibrational mode. Change this value.
mode_index = 0
print(f"Showing mode {mode_index}: {frequencies[mode_index]:.2f} cm^-1")
show_mode(opt_molecule, mode_index, amplitude=0.45, frames=32).show()
print(normal_modes[mode_index])

Showing mode 0: 115.39 cm^-1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

[[ 0.0146229   0.04909266  0.02091396]
 [ 0.01195333  0.04018638  0.01712348]
 [ 0.00777821  0.02614307  0.0111558 ]
 [ 0.01031226  0.03460798  0.01474185]
 [ 0.00984283  0.03308745  0.01410165]
 [-0.02510699 -0.08439436 -0.03598546]
 [-0.00603748 -0.02025967 -0.00862008]
 [-0.0156616  -0.05258871 -0.02239175]
 [-0.00747921 -0.02510176 -0.01069828]
 [ 0.00799788  0.02688502  0.01144594]
 [ 0.01560208  0.05226539  0.02234177]
 [ 0.00582006  0.01946883  0.00825971]
 [ 0.10600817  0.36491908  0.15559892]
 [ 0.35274499 -0.4929827  -0.15509007]
 [-0.56419027 -0.22635849 -0.15158412]
 [-0.01214342 -0.04076115 -0.01735448]
 [-0.02936895 -0.09862262 -0.04200493]
 [-0.01420868 -0.04773801 -0.02035169]
 [ 0.01277334  0.04291706  0.01827085]]
